# Efficient Daily Pre-Match AWS Ratings

这个 notebook 是高效版本，适合更大的 `match_heatmap.pkl`。

## 核心逻辑

1. **AWS 按比赛日更新**  
   同一天不同时段的比赛只 fit 一次 AWS model。

2. **避免 data leakage**  
   对于某个比赛日 `D`，训练数据只使用：

```python
players_clean["Match_Day"] < D
```

也就是当天早场比赛不会进入当天晚场比赛的 AWS 计算。

3. **no-history 球员不进入 regression fitting**  
   如果某个当天参赛球员在过去 2 年窗口内没有任何非空 `Player_rating`：

```python
该球员不加入 fitting
```

先用历史球员 fit 完 AWS model，然后使用 pseudo match 公式反推该球员的 AWS：

```python
pseudo_rating = historical league average
IsHome = 0.5

AWS_player = pseudo_rating - intercept - 0.5 * home_coef - league_coef
```

所以最终返回的是根据 fitted model 和 pseudo match 算出来的 AWS，不是直接返回 pseudo rating。

4. **多进程**  
   多进程粒度是 `Match_Day`。每个比赛日独立 fit 一个 AWS model。

5. **大 pkl 优化**  
   多进程只传递必要的 metadata columns，不传 heatmap 大对象。


In [1]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
import os
import time
from typing import Any

import numpy as np
import pandas as pd
from scipy import sparse
from scipy.sparse.linalg import spsolve, lsmr


# =========================
# Config
# =========================

@dataclass(frozen=True)
class AWSParams:
    psi_years: float = 2.0
    phi: float = 0.0062
    omega: float = 7.0
    ridge: float = 1e-8
    solver: str = "normal_eq"  # "normal_eq" is faster; "lsmr" is closer to the original notebook.
    lsmr_atol: float = 1e-8
    lsmr_btol: float = 1e-8
    lsmr_maxiter: int | None = None


# =========================
# Data cleaning
# =========================

def clean_players(players_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only historical rows with non-null Player_rating.
    These rows are the real observations used in AWS model fitting.
    """
    required = {"Match_Date", "League", "IsHome", "Player", "Player_ID", "Player_rating"}
    missing = required.difference(players_raw.columns)
    if missing:
        raise ValueError(f"players file missing required columns: {sorted(missing)}")

    players = players_raw.dropna(
        subset=["Player_rating", "Match_Date", "League", "Player_ID"]
    ).copy()

    players["Match_Date"] = pd.to_datetime(players["Match_Date"])
    players["Match_Day"] = players["Match_Date"].dt.normalize()
    players["IsHome"] = players["IsHome"].astype(float)
    players["Player_ID"] = players["Player_ID"].astype(str).str.replace(r"\.0$", "", regex=True)
    players["League"] = players["League"].astype(str)

    # Keep only the columns needed by fitting to reduce memory.
    cols = ["Match_Date", "Match_Day", "League", "IsHome", "Player", "Player_ID", "Player_rating"]
    return players[cols].copy()


def prepare_match_frame(match_df: pd.DataFrame) -> pd.DataFrame:
    """
    Create a small metadata frame from a potentially huge match dataframe.
    Do NOT pass heatmap columns into multiprocessing workers.
    """
    required = {
        "League",
        "Match_Date",
        "Starting_11(Home)", "Substitute(Home)",
        "Starting_11(Away)", "Substitute(Away)",
    }
    missing = required.difference(match_df.columns)
    if missing:
        raise ValueError(f"match dataframe missing required columns: {sorted(missing)}")

    cols = [
        "League",
        "Match_Date",
        "Starting_11(Home)", "Substitute(Home)",
        "Starting_11(Away)", "Substitute(Away)",
    ]

    meta = match_df[cols].copy()
    meta["Match_Date"] = pd.to_datetime(meta["Match_Date"])
    meta["Match_Day"] = meta["Match_Date"].dt.normalize()

    return meta


def normalize_player_id(pid: Any) -> str | None:
    if pd.isna(pid):
        return None

    try:
        return str(int(pid))
    except Exception:
        return str(pid)


def collect_match_player_ids(row: pd.Series) -> list[str]:
    """
    Merge Home/Away and starting/substitute player IDs into one ordered unique list.
    """
    player_cols = [
        "Starting_11(Home)",
        "Substitute(Home)",
        "Starting_11(Away)",
        "Substitute(Away)",
    ]

    player_ids: list[str] = []
    seen: set[str] = set()

    for col in player_cols:
        values = row[col]

        if values is None or (isinstance(values, float) and pd.isna(values)):
            continue

        for pid in list(values):
            key = normalize_player_id(pid)

            if key is not None and key not in seen:
                seen.add(key)
                player_ids.append(key)

    return player_ids


def make_current_day_player_frame(day_meta: pd.DataFrame) -> pd.DataFrame:
    """
    For one match day, collect all players appearing in the match dataframe.
    These players are used only for post-fit AWS calculation if they have no history.
    They are NOT inserted into the regression fitting.
    """
    rows = []

    for _, row in day_meta.iterrows():
        league = str(row["League"])

        for pid in collect_match_player_ids(row):
            rows.append({"Player_ID": str(pid), "League": league})

    if not rows:
        return pd.DataFrame(columns=["Player_ID", "League"])

    return pd.DataFrame(rows).drop_duplicates("Player_ID")


# =========================
# AWS fitting
# =========================

def build_training_frame_by_day(
    players_clean: pd.DataFrame,
    match_day: pd.Timestamp,
    params: AWSParams,
) -> tuple[pd.DataFrame, pd.Series, float]:
    """
    Build the AWS regression frame for one calendar match day.

    Important:
    - Fit once per Match_Day.
    - Use historical player ratings strictly before the current Match_Day.
    - Add pseudo matches only for players who have historical ratings in the 2-year window.
    - Current-day no-history players are NOT added into this training frame.
    """
    day = pd.Timestamp(match_day).normalize()
    start_day = day - pd.Timedelta(days=int(round(params.psi_years * 365.25)))

    real = players_clean[
        (players_clean["Match_Day"] < day) &
        (players_clean["Match_Day"] >= start_day)
    ].copy()

    if real.empty:
        raise ValueError(f"No valid historical Player_rating before match day {day.date()}.")

    days_old = (day - real["Match_Day"]).dt.days.astype(float)
    real["sample_weight"] = np.exp(-params.phi * days_old / 3.5)
    real["is_pseudo"] = False

    latest_hist_players = (
        real.sort_values("Match_Date")
        .groupby("Player_ID", as_index=False)
        .tail(1)[["Player_ID", "Player", "League"]]
    )

    league_avg = real.groupby("League")["Player_rating"].mean()
    global_avg = float(real["Player_rating"].mean())

    pseudo = latest_hist_players.copy()
    pseudo["Match_Date"] = day
    pseudo["Match_Day"] = day
    pseudo["IsHome"] = 0.5
    pseudo["Player_rating"] = pseudo["League"].map(league_avg).fillna(global_avg).astype(float)
    pseudo["sample_weight"] = params.omega
    pseudo["is_pseudo"] = True

    cols = [
        "Match_Date", "Match_Day", "League", "IsHome", "Player", "Player_ID",
        "Player_rating", "sample_weight", "is_pseudo",
    ]

    train = pd.concat([real[cols], pseudo[cols]], ignore_index=True)

    return train, league_avg, global_avg


def solve_weighted_regression(X: sparse.csr_matrix, y: np.ndarray, w: np.ndarray, params: AWSParams) -> np.ndarray:
    """
    Solve weighted least squares.

    normal_eq:
        Faster in this workflow.
        Solves (X'WX + ridge I) beta = X'Wy.

    lsmr:
        Closer to the original AWS notebook style.
        Solves min ||sqrt(W)(X beta - y)||.
    """
    solver = params.solver.lower()

    if solver == "normal_eq":
        Xw = X.multiply(w[:, None])
        A = X.T @ Xw
        b = X.T @ (y * w)

        if params.ridge and params.ridge > 0:
            A = A + sparse.eye(A.shape[0], format="csr") * params.ridge

        return spsolve(A.tocsc(), b)

    if solver == "lsmr":
        sqrt_w = np.sqrt(w)
        X_weighted = X.multiply(sqrt_w[:, None])
        y_weighted = y * sqrt_w

        return lsmr(
            X_weighted,
            y_weighted,
            atol=params.lsmr_atol,
            btol=params.lsmr_btol,
            maxiter=params.lsmr_maxiter,
        )[0]

    raise ValueError("params.solver must be either 'normal_eq' or 'lsmr'.")


def fit_aws_model_for_day(
    players_clean: pd.DataFrame,
    match_day: pd.Timestamp,
    current_day_players: pd.DataFrame,
    params: AWSParams,
) -> dict[str, Any]:
    """
    Fit AWS model once for one match day.

    For no-history players:
    - They are not used in fitting.
    - After fitting, construct their pseudo match:
        pseudo_rating = historical league average
        IsHome = 0.5
    - Use the fitted model coefficients to infer:
        AWS_player = pseudo_rating - intercept - 0.5 * home_coef - league_coef
    """
    day = pd.Timestamp(match_day).normalize()

    train, league_avg, global_avg = build_training_frame_by_day(
        players_clean=players_clean,
        match_day=day,
        params=params,
    )

    player_codes, player_index = pd.factorize(train["Player_ID"], sort=True)
    league_codes, league_index = pd.factorize(train["League"], sort=True)

    n = len(train)
    n_players = len(player_index)
    n_leagues = len(league_index)

    intercept_col = sparse.csr_matrix(np.ones((n, 1)))
    home_col = sparse.csr_matrix(train[["IsHome"]].to_numpy(dtype=float))

    player_x = sparse.csr_matrix(
        (np.ones(n), (np.arange(n), player_codes)),
        shape=(n, n_players),
    )

    league_x = sparse.csr_matrix(
        (np.ones(n), (np.arange(n), league_codes)),
        shape=(n, n_leagues),
    )

    X = sparse.hstack([intercept_col, home_col, player_x, league_x], format="csr")

    y = train["Player_rating"].to_numpy(dtype=float)
    w = train["sample_weight"].to_numpy(dtype=float)

    coef = solve_weighted_regression(X, y, w, params)

    intercept = float(coef[0])
    home_coef = float(coef[1])
    player_coef = coef[2: 2 + n_players]
    league_coef = coef[2 + n_players:]

    player_index_str = pd.Index(player_index.astype(str))
    league_index_str = pd.Index(league_index.astype(str))

    aws_map: dict[str, float] = dict(zip(player_index_str, player_coef.astype(float)))
    league_coef_map: dict[str, float] = dict(zip(league_index_str, league_coef.astype(float)))

    historical_player_ids = set(player_index_str)
    pseudo_formula_ids: set[str] = set()

    # Compute AWS for current-day players with no historical rating in the 2-year window.
    # They are NOT part of fitting. Their AWS is inferred from the fitted model and their pseudo match.
    if current_day_players is not None and len(current_day_players) > 0:
        for _, row in current_day_players.iterrows():
            pid = str(row["Player_ID"])
            league = str(row["League"])

            if pid in aws_map:
                continue

            pseudo_rating = league_avg.get(league, global_avg)

            if pd.isna(pseudo_rating):
                pseudo_rating = global_avg

            league_effect = league_coef_map.get(league, 0.0)

            pseudo_aws = float(pseudo_rating) - intercept - 0.5 * home_coef - float(league_effect)

            aws_map[pid] = float(pseudo_aws)
            pseudo_formula_ids.add(pid)

    return {
        "match_day": day,
        "aws_map": aws_map,
        "pseudo_formula_ids": pseudo_formula_ids,
        "n_model_players": len(historical_player_ids),
        "n_current_day_players": 0 if current_day_players is None else len(current_day_players),
        "n_pseudo_formula_players": len(pseudo_formula_ids),
    }


# =========================
# Parallel daily fitting
# =========================

_GLOBAL_PLAYERS_CLEAN = None
_GLOBAL_PARAMS = None


def _init_worker(players_clean: pd.DataFrame, params: AWSParams):
    global _GLOBAL_PLAYERS_CLEAN, _GLOBAL_PARAMS
    _GLOBAL_PLAYERS_CLEAN = players_clean
    _GLOBAL_PARAMS = params


def _fit_one_day_worker(task: tuple[pd.Timestamp, list[dict[str, Any]]]) -> dict[str, Any]:
    global _GLOBAL_PLAYERS_CLEAN, _GLOBAL_PARAMS

    if _GLOBAL_PLAYERS_CLEAN is None or _GLOBAL_PARAMS is None:
        raise RuntimeError("Worker globals are not initialised.")

    day, current_players_records = task
    current_day_players = pd.DataFrame(current_players_records)

    return fit_aws_model_for_day(
        players_clean=_GLOBAL_PLAYERS_CLEAN,
        match_day=pd.Timestamp(day),
        current_day_players=current_day_players,
        params=_GLOBAL_PARAMS,
    )


def build_daily_tasks(match_meta: pd.DataFrame) -> list[tuple[pd.Timestamp, list[dict[str, Any]]]]:
    tasks = []

    for day, day_meta in match_meta.groupby("Match_Day", sort=True):
        current_day_players = make_current_day_player_frame(day_meta)
        tasks.append((pd.Timestamp(day).normalize(), current_day_players.to_dict("records")))

    return tasks


def fit_all_days(
    match_meta: pd.DataFrame,
    players_clean: pd.DataFrame,
    params: AWSParams,
    n_jobs: int = 1,
    verbose: bool = True,
) -> dict[pd.Timestamp, dict[str, Any]]:
    """
    Fit AWS models for all unique match days.

    n_jobs=1:
        sequential; safest in notebooks.

    n_jobs>1:
        multiprocessing by match day.
        Recommended for running as a .py script or a stable local notebook environment.
    """
    tasks = build_daily_tasks(match_meta)
    total = len(tasks)

    if verbose:
        print(f"Number of exact match timestamps: {match_meta['Match_Date'].nunique()}")
        print(f"Number of unique match days: {total}")
        print(f"n_jobs: {n_jobs}")

    start_time = time.time()
    results: dict[pd.Timestamp, dict[str, Any]] = {}

    if n_jobs == 1:
        for i, (day, records) in enumerate(tasks, start=1):
            res = fit_aws_model_for_day(
                players_clean=players_clean,
                match_day=day,
                current_day_players=pd.DataFrame(records),
                params=params,
            )
            results[pd.Timestamp(day).normalize()] = res

            if verbose and (i == 1 or i % 10 == 0 or i == total):
                print(
                    f"fitted {i}/{total} days | day={pd.Timestamp(day).date()} | "
                    f"pseudo_formula_players={res['n_pseudo_formula_players']} | "
                    f"elapsed={time.time() - start_time:.1f}s"
                )

    else:
        max_workers = max(1, int(n_jobs))

        with ProcessPoolExecutor(
            max_workers=max_workers,
            initializer=_init_worker,
            initargs=(players_clean, params),
        ) as executor:
            future_to_day = {
                executor.submit(_fit_one_day_worker, task): pd.Timestamp(task[0]).normalize()
                for task in tasks
            }

            done = 0
            for future in as_completed(future_to_day):
                day = future_to_day[future]
                res = future.result()
                results[day] = res
                done += 1

                if verbose and (done == 1 or done % 10 == 0 or done == total):
                    print(
                        f"fitted {done}/{total} days | latest_day={day.date()} | "
                        f"pseudo_formula_players={res['n_pseudo_formula_players']} | "
                        f"elapsed={time.time() - start_time:.1f}s"
                    )

    return results


# =========================
# Add dictionary column to match dataframe
# =========================

def add_aws_dictionary_columns(
    match_df: pd.DataFrame,
    match_meta: pd.DataFrame,
    day_results: dict[pd.Timestamp, dict[str, Any]],
) -> pd.DataFrame:
    """
    Add:
    - PreMatch_AWS_Rating: {player_id: prematch AWS}
    - PreMatch_AWS_PseudoFormula_Count: number of players whose AWS was inferred using pseudo formula
    - PreMatch_AWS_Missing_Count: number of missing players; should be zero
    """
    out = match_df.copy()
    out["Match_Date"] = pd.to_datetime(out["Match_Date"])
    out["Match_Day"] = out["Match_Date"].dt.normalize()

    prematch_col = []
    pseudo_formula_counts = []
    missing_counts = []

    # Iterate over small meta, but collect dictionaries aligned with original row order.
    for idx, row in match_meta.iterrows():
        day = pd.Timestamp(row["Match_Day"]).normalize()
        player_ids = collect_match_player_ids(row)

        res = day_results[day]
        aws_map = res["aws_map"]
        pseudo_formula_ids = res["pseudo_formula_ids"]

        ratings = {}
        n_pseudo_formula = 0
        n_missing = 0

        for pid in player_ids:
            pid = str(pid)

            if pid in aws_map:
                ratings[pid] = float(aws_map[pid])

                if pid in pseudo_formula_ids:
                    n_pseudo_formula += 1
            else:
                ratings[pid] = None
                n_missing += 1

        prematch_col.append((idx, ratings))
        pseudo_formula_counts.append((idx, n_pseudo_formula))
        missing_counts.append((idx, n_missing))

    # Assign by index to preserve original dataframe ordering.
    prematch_series = pd.Series(dict(prematch_col), name="PreMatch_AWS_Rating")
    pseudo_series = pd.Series(dict(pseudo_formula_counts), name="PreMatch_AWS_PseudoFormula_Count")
    missing_series = pd.Series(dict(missing_counts), name="PreMatch_AWS_Missing_Count")

    out["PreMatch_AWS_Rating"] = prematch_series.reindex(out.index)
    out["PreMatch_AWS_PseudoFormula_Count"] = pseudo_series.reindex(out.index).astype(int)
    out["PreMatch_AWS_Missing_Count"] = missing_series.reindex(out.index).astype(int)

    return out


def process_match_file(
    match_pkl_path: str | Path,
    players_csv_path: str | Path,
    output_pkl_path: str | Path,
    output_preview_csv_path: str | Path | None = None,
    params: AWSParams | None = None,
    n_jobs: int | None = None,
    verbose: bool = True,
) -> pd.DataFrame:
    """
    Full pipeline.

    Recommended n_jobs:
    - For notebook: start with n_jobs=1.
    - For script/local machine: n_jobs=min(4, os.cpu_count()-1).
    """
    if params is None:
        params = AWSParams()

    if n_jobs is None:
        cpu = os.cpu_count() or 2
        n_jobs = max(1, min(4, cpu - 1))

    match_pkl_path = Path(match_pkl_path)
    players_csv_path = Path(players_csv_path)
    output_pkl_path = Path(output_pkl_path)

    if verbose:
        print("Loading match dataframe...")
    match_df = pd.read_pickle(match_pkl_path)

    if verbose:
        print("Loading players dataframe...")
    players_raw = pd.read_csv(players_csv_path, parse_dates=["Match_Date"])

    players_clean = clean_players(players_raw)
    match_meta = prepare_match_frame(match_df)

    if verbose:
        print("Fitting daily AWS models...")
    day_results = fit_all_days(
        match_meta=match_meta,
        players_clean=players_clean,
        params=params,
        n_jobs=n_jobs,
        verbose=verbose,
    )

    if verbose:
        print("Adding AWS dictionary column to match dataframe...")
    out = add_aws_dictionary_columns(
        match_df=match_df,
        match_meta=match_meta,
        day_results=day_results,
    )

    if verbose:
        print(f"Saving pkl to: {output_pkl_path}")
    out.to_pickle(output_pkl_path)

    if output_preview_csv_path is not None:
        output_preview_csv_path = Path(output_preview_csv_path)
        preview_cols = [c for c in out.columns if "Heatmap" not in c]

        if verbose:
            print(f"Saving preview csv to: {output_preview_csv_path}")

        out[preview_cols].to_csv(output_preview_csv_path, index=True)

    if verbose:
        print("\nDiagnostics:")
        print(out[["PreMatch_AWS_PseudoFormula_Count", "PreMatch_AWS_Missing_Count"]].describe())

    return out

## File paths

把 `MATCH_PKL` 改成你的大 pkl 路径即可。  
如果在 notebook 里多进程不稳定，先用 `N_JOBS = 1`；如果在本地 `.py` 里跑，可以设成 4 或更多。


In [19]:
MATCH_PKL = "match_heatmap.pkl"
PLAYERS_CSV = "players.csv"

OUTPUT_PKL = "match_heatmap_fraction_with_flat_prematch_aws.pkl"
OUTPUT_PREVIEW_CSV = "match_heatmap_fraction_with_efficient_daily_aws_preview.csv"

# Notebook 里建议先从 1 开始；确认没问题后再改成 2/4。
N_JOBS = 1

params = AWSParams(
    solver="normal_eq",  # faster; use "lsmr" to be closer to the original AWS notebook
)

MATCH_PKL, PLAYERS_CSV, OUTPUT_PKL

('match_heatmap.pkl',
 'players.csv',
 'match_heatmap_fraction_with_flat_prematch_aws.pkl')

## Run pipeline

In [20]:
%%time
match_with_aws = process_match_file(
    match_pkl_path=MATCH_PKL,
    players_csv_path=PLAYERS_CSV,
    output_pkl_path=OUTPUT_PKL,
    output_preview_csv_path=OUTPUT_PREVIEW_CSV,
    params=params,
    n_jobs=N_JOBS,
    verbose=True,
)

match_with_aws[[
    "Match_Date",
    "Match_Day",
    "PreMatch_AWS_Rating",
    "PreMatch_AWS_PseudoFormula_Count",
    "PreMatch_AWS_Missing_Count",
]].head()

Loading match dataframe...
Loading players dataframe...
Fitting daily AWS models...
Number of exact match timestamps: 9098
Number of unique match days: 1651
n_jobs: 1
fitted 1/1651 days | day=2017-08-11 | pseudo_formula_players=7 | elapsed=0.1s
fitted 10/1651 days | day=2017-08-27 | pseudo_formula_players=78 | elapsed=1.2s
fitted 20/1651 days | day=2017-09-20 | pseudo_formula_players=62 | elapsed=2.5s
fitted 30/1651 days | day=2017-10-14 | pseudo_formula_players=49 | elapsed=3.8s
fitted 40/1651 days | day=2017-10-28 | pseudo_formula_players=37 | elapsed=5.0s
fitted 50/1651 days | day=2017-11-24 | pseudo_formula_players=4 | elapsed=6.3s
fitted 60/1651 days | day=2017-12-08 | pseudo_formula_players=5 | elapsed=7.8s
fitted 70/1651 days | day=2017-12-19 | pseudo_formula_players=0 | elapsed=9.1s
fitted 80/1651 days | day=2017-12-31 | pseudo_formula_players=1 | elapsed=10.4s
fitted 90/1651 days | day=2018-01-13 | pseudo_formula_players=34 | elapsed=11.6s
fitted 100/1651 days | day=2018-01-26

fitted 990/1651 days | day=2022-12-26 | pseudo_formula_players=13 | elapsed=127.8s
fitted 1000/1651 days | day=2023-01-05 | pseudo_formula_players=3 | elapsed=129.1s
fitted 1010/1651 days | day=2023-01-16 | pseudo_formula_players=3 | elapsed=130.4s
fitted 1020/1651 days | day=2023-01-28 | pseudo_formula_players=26 | elapsed=131.7s
fitted 1030/1651 days | day=2023-02-10 | pseudo_formula_players=9 | elapsed=132.9s
fitted 1040/1651 days | day=2023-02-25 | pseudo_formula_players=31 | elapsed=134.2s
fitted 1050/1651 days | day=2023-03-12 | pseudo_formula_players=39 | elapsed=135.5s
fitted 1060/1651 days | day=2023-04-04 | pseudo_formula_players=5 | elapsed=136.8s
fitted 1070/1651 days | day=2023-04-21 | pseudo_formula_players=8 | elapsed=138.1s
fitted 1080/1651 days | day=2023-05-02 | pseudo_formula_players=13 | elapsed=139.4s
fitted 1090/1651 days | day=2023-05-15 | pseudo_formula_players=1 | elapsed=140.7s
fitted 1100/1651 days | day=2023-05-28 | pseudo_formula_players=56 | elapsed=142.0s

,Match_Date,Match_Day,PreMatch_AWS_Rating,PreMatch_AWS_PseudoFormula_Count,PreMatch_AWS_Missing_Count
16535,2017-08-11 18:00:00,2017-08-11,"{'140569': -0.13428506974468565, '80465': 0.04...",3,0
767,2017-08-11 19:45:00,2017-08-11,"{'6775': 0.3569352221829727, '110260': 0.25701...",1,0
16562,2017-08-11 19:45:00,2017-08-11,"{'82622': -0.07756774199580838, '289018': 0.05...",3,0
1092,2017-08-12 12:30:00,2017-08-12,"{'10133': 0.27682036591019027, '32323': 0.1032...",2,0
887,2017-08-12 15:00:00,2017-08-12,"{'110189': 0.39545046616721574, '6105': 0.5218...",1,0


## Checks

In [7]:
print("Missing count total:", match_with_aws["PreMatch_AWS_Missing_Count"].sum())

display(match_with_aws[[
    "PreMatch_AWS_PseudoFormula_Count",
    "PreMatch_AWS_Missing_Count",
]].describe())

print("\nExample first row:")
match_with_aws["PreMatch_AWS_Rating"].iloc[0]

Missing count total: 0


,PreMatch_AWS_PseudoFormula_Count,PreMatch_AWS_Missing_Count
count,50.000000,50.0
mean,2.000000,0.0
std,1.641304,0.0
min,0.000000,0.0
25%,1.000000,0.0
50%,2.000000,0.0
75%,3.000000,0.0
max,6.000000,0.0



Example first row:


{'70483': 0.10695019568155109,
 '301441': 0.1675546915120111,
 '355609': 0.3068990693763751,
 '302312': 0.15700118221514378,
 '330549': 0.08541999881943754,
 '354291': 0.0658389347135919,
 '382590': 0.05477504715357165,
 '401073': 0.4731632627353892,
 '322418': 0.30671392401958514,
 '371012': 0.23345515261592162,
 '141469': 0.265464969813157,
 '347340': -0.11025460147976053,
 '439636': -0.11939377724588937,
 '369542': -0.19784842163394512,
 '360941': 0.19435728062417362,
 '302313': -0.1652106884091198,
 '364235': -0.04282699190571717,
 '307564': 0.04466202316543264,
 '423347': -0.06473678129406192,
 '342811': 0.07043808091404681,
 '425235': 0.18171082415568962,
 '134373': -0.11582448023745909,
 '344639': 0.35100495939016285,
 '371761': 0.435685445539528,
 '424514': 0.16758517890866823,
 '437449': 0.12043984836598826,
 '345441': -0.057204885110952036,
 '443064': 0.005306361039534754,
 '422818': 0.04434276453228826,
 '398393': 0.19202367942313892,
 '494670': -0.023206804997999788,
 '1098

## For large pkl

建议：

```python
N_JOBS = 4
```

如果你的机器内存较大，可以提高到：

```python
N_JOBS = 6 or 8
```

如果 notebook 多进程报错，直接运行我生成的 `.py` 脚本更稳定：

```bash
python efficient_daily_prematch_aws.py
```

或者在脚本里修改 `MATCH_PKL` 为你的大文件路径。


In [21]:
match_with_aws

,League,Match_Date,Home_Team,Away_Team,Home_Team_ID,Away_Team_ID,Score,Home_Score,Away_Score,Score_Diff,...,Starting_11_Heatmap(Away),Substitute_Heatmap(Away),Starting_11_Touches(Home),Substitute_Touches(Home),Starting_11_Touches(Away),Substitute_Touches(Away),Match_Day,PreMatch_AWS_Rating,PreMatch_AWS_PseudoFormula_Count,PreMatch_AWS_Missing_Count
16535,France_Ligue1,2017-08-11 18:00:00,Nice,Troyes,613,229,1 : 2,1,2,-1,...,"{68861: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.027777778...","{299164: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",902,48,482,19,2017-08-11,"{'140569': -0.13428506974468565, '80465': 0.04...",3,0
767,England_Premier_League,2017-08-11 19:45:00,Arsenal,Leicester,13,14,4 : 3,4,3,1,...,"{19545: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.022727273...","{243552: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",826,33,444,15,2017-08-11,"{'6775': 0.3569352221829727, '110260': 0.25701...",1,0
16562,France_Ligue1,2017-08-11 19:45:00,Rennes,Lyon,313,228,1 : 2,1,2,-1,...,"{71833: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.057142857...","{322095: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",618,14,605,30,2017-08-11,"{'82622': -0.07756774199580838, '289018': 0.05...",3,0
1092,England_Premier_League,2017-08-12 12:30:00,Watford,Liverpool,27,26,3 : 3,3,3,0,...,"{52197: [[0.0, 0.0, 0.0, 0.020408163, 0.0, 0.0...","{124688: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",514,81,670,12,2017-08-12,"{'10133': 0.27682036591019027, '32323': 0.1032...",2,0
887,England_Premier_League,2017-08-12 15:00:00,Everton,Stoke,31,96,1 : 0,1,0,1,...,"{107395: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.09090909...","{29809: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0...",628,74,491,30,2017-08-12,"{'110189': 0.39545046616721574, '6105': 0.5218...",1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12144,Italy_Serie_A,2026-05-24 19:45:00,Verona,Roma,76,84,0 : 2,0,2,-2,...,"{331422: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.05882353...","{369570: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",384,90,700,83,2026-05-24,"{'142425': -0.0802786534349164, '414212': 0.13...",5,0
11984,Italy_Serie_A,2026-05-24 19:45:00,Lecce,Genoa,79,278,1 : 0,1,0,1,...,"{91169: [[0.0, 0.0, 0.0, 0.0, 0.018867925, 0.0...","{523963: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",386,31,584,95,2026-05-24,"{'118163': 0.15598015630460446, '444938': 0.05...",7,0
11770,Italy_Serie_A,2026-05-24 19:45:00,AC Milan,Cagliari,80,78,1 : 2,1,2,-1,...,"{362780: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.02439024...","{446991: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",540,128,622,64,2026-05-24,"{'141646': 0.21009869959333577, '337443': 0.15...",7,0
8336,Spain_Laliga,2026-05-24 20:00:00,Villarreal,Atletico,839,63,5 : 1,5,1,4,...,"{126278: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.088...","{402046: [[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",604,69,583,107,2026-05-24,"{'396147': 0.07653229025938771, '448936': 0.30...",2,0
